# audio_conversion

> Audio conversion service for browser-compatible CBR audio

In [ ]:
#| default_exp services.audio_conversion

In [ ]:
#| export
import subprocess
import hashlib
from pathlib import Path
from typing import Optional

## Overview

Browser HTML5 Audio has inaccurate seeking with VBR (Variable Bit Rate) MP3 files.
This service converts audio to CBR (Constant Bit Rate) format for precise seeking.

The conversion is cached in `.cjm/data/audio_cache/` to avoid repeated processing.

In [ ]:
#| export
# Debug flag (set True for conversion tracing)
DEBUG_AUDIO_CONVERSION = True

## Cache Path Generation

Generates a deterministic cache path based on the source file path.
Uses a hash of the absolute path to create a unique filename.

In [ ]:
#| export
def get_cache_path(
    source_path: str,  # Path to the source audio file
    cache_dir: Path,  # Directory for cached files
) -> Path:  # Path where cached CBR file should be stored
    """Generate cache path for a converted audio file."""
    # Create hash from absolute path
    source_abs = Path(source_path).resolve()
    path_hash = hashlib.md5(str(source_abs).encode()).hexdigest()[:12]
    
    # Use original filename stem + hash for readability
    stem = source_abs.stem
    cache_name = f"{stem}_{path_hash}_cbr.mp3"
    # cache_name = f"{stem}_{path_hash}_cbr.wav"
    
    return cache_dir / cache_name

In [ ]:
# Test get_cache_path
cache_dir = Path("/tmp/audio_cache")
source = "/path/to/audio/02 - 1. Laying Plans.mp3"

cache_path = get_cache_path(source, cache_dir)
print(f"Source: {source}")
print(f"Cache path: {cache_path}")

# Verify deterministic
cache_path2 = get_cache_path(source, cache_dir)
assert cache_path == cache_path2, "Cache path should be deterministic"
print("Cache path is deterministic: OK")

Source: /path/to/audio/02 - 1. Laying Plans.mp3
Cache path: /tmp/audio_cache/02 - 1. Laying Plans_44dbfab7b2c3_cbr.mp3
Cache path is deterministic: OK


## Check if Conversion Needed

Checks if a cached CBR version exists and is newer than the source file.

In [ ]:
#| export
def needs_conversion(
    source_path: str,  # Path to the source audio file
    cache_path: Path,  # Path to the cached CBR file
) -> bool:  # True if conversion is needed
    """Check if audio needs conversion (cache missing or stale)."""
    if not cache_path.exists():
        if DEBUG_AUDIO_CONVERSION:
            print(f"[AUDIO_CONV] Cache miss: {cache_path}")
        return True
    
    # Check if source is newer than cache
    source_mtime = Path(source_path).stat().st_mtime
    cache_mtime = cache_path.stat().st_mtime
    
    if source_mtime > cache_mtime:
        if DEBUG_AUDIO_CONVERSION:
            print(f"[AUDIO_CONV] Cache stale: {cache_path}")
        return True
    
    if DEBUG_AUDIO_CONVERSION:
        print(f"[AUDIO_CONV] Cache hit: {cache_path}")
    return False

## FFmpeg Conversion

Converts audio to CBR MP3 using ffmpeg. Uses settings optimized for
browser compatibility and accurate seeking:

- 192 kbps CBR (constant bit rate)
- 44100 Hz sample rate
- Stereo output

In [ ]:
#| export
def convert_to_cbr(
    source_path: str,  # Path to the source audio file
    output_path: Path,  # Path for the output CBR file
    bitrate: str = "192k",  # Target bitrate (e.g., "128k", "192k", "320k")
    sample_rate: int = 44100,  # Output sample rate in Hz
) -> bool:  # True if conversion succeeded
    """Convert audio file to CBR MP3 using ffmpeg."""
    # Ensure output directory exists
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    cmd = [
        "ffmpeg",
        "-y",  # Overwrite output
        "-i", str(source_path),
        "-b:a", bitrate,  # Constant bitrate
        "-ar", str(sample_rate),  # Sample rate
        # "-ac", "2",  # Stereo
        str(output_path)
    ]
    
    if DEBUG_AUDIO_CONVERSION:
        print(f"[AUDIO_CONV] Running: {' '.join(cmd)}")
    
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=True
        )
        if DEBUG_AUDIO_CONVERSION:
            print(f"[AUDIO_CONV] Conversion complete: {output_path}")
        return True
    except subprocess.CalledProcessError as e:
        if DEBUG_AUDIO_CONVERSION:
            print(f"[AUDIO_CONV] Conversion failed: {e.stderr}")
        return False
    except FileNotFoundError:
        if DEBUG_AUDIO_CONVERSION:
            print("[AUDIO_CONV] ffmpeg not found in PATH")
        return False

## Main Entry Point

The primary function for ensuring a CBR version of an audio file exists.
Returns the path to use for browser playback.

In [ ]:
#| export
def ensure_cbr_audio(
    source_path: str,  # Path to the source audio file
    cache_dir: Optional[Path] = None,  # Cache directory (default: .cjm/data/audio_cache)
    project_root: Optional[Path] = None,  # Project root for default cache dir
) -> Optional[str]:  # Path to CBR audio file, or None if conversion failed
    """Ensure a CBR version of the audio file exists for browser playback."""
    source = Path(source_path)
    if not source.exists():
        if DEBUG_AUDIO_CONVERSION:
            print(f"[AUDIO_CONV] Source file not found: {source_path}")
        return None
    
    # Determine cache directory
    if cache_dir is None:
        if project_root is None:
            # Default to source file's directory
            cache_dir = source.parent / ".audio_cache"
        else:
            cache_dir = project_root / ".cjm" / "data" / "audio_cache"
    
    cache_path = get_cache_path(source_path, cache_dir)
    
    # Check if conversion needed
    if needs_conversion(source_path, cache_path):
        if DEBUG_AUDIO_CONVERSION:
            print(f"[AUDIO_CONV] Converting: {source_path}")
        
        if not convert_to_cbr(source_path, cache_path):
            # Conversion failed, fall back to original
            if DEBUG_AUDIO_CONVERSION:
                print(f"[AUDIO_CONV] Falling back to original: {source_path}")
            return source_path
    
    return str(cache_path)

## Tests

Test the audio conversion functionality. Requires ffmpeg to be installed.

In [ ]:
#| eval: false
# Test with a real audio file
from pathlib import Path
import tempfile

# Test audio file
test_audio = Path("/mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-transcription-plugin-voxtral-hf/test_files/short_test_audio.mp3")

if test_audio.exists():
    # Use temp directory for cache
    with tempfile.TemporaryDirectory() as tmpdir:
        cache_dir = Path(tmpdir)
        
        # First call should convert
        print("First call (should convert):")
        result = ensure_cbr_audio(str(test_audio), cache_dir=cache_dir)
        print(f"  Result: {result}")
        
        if result:
            result_path = Path(result)
            print(f"  Exists: {result_path.exists()}")
            print(f"  Size: {result_path.stat().st_size / 1024:.1f} KB")
        
        # Second call should use cache
        print("\nSecond call (should use cache):")
        result2 = ensure_cbr_audio(str(test_audio), cache_dir=cache_dir)
        print(f"  Result: {result2}")
        assert result == result2, "Should return same cached path"
        print("  Cache reuse: OK")
else:
    print(f"Test audio not found: {test_audio}")

First call (should convert):
[AUDIO_CONV] Cache miss: /tmp/tmpz0lle0cj/short_test_audio_b64fa473a841_cbr.mp3
[AUDIO_CONV] Converting: /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-transcription-plugin-voxtral-hf/test_files/short_test_audio.mp3
[AUDIO_CONV] Running: ffmpeg -y -i /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-transcription-plugin-voxtral-hf/test_files/short_test_audio.mp3 -b:a 192k -ar 44100 /tmp/tmpz0lle0cj/short_test_audio_b64fa473a841_cbr.mp3
[AUDIO_CONV] Conversion complete: /tmp/tmpz0lle0cj/short_test_audio_b64fa473a841_cbr.mp3
  Result: /tmp/tmpz0lle0cj/short_test_audio_b64fa473a841_cbr.mp3
  Exists: True
  Size: 657.6 KB

Second call (should use cache):
[AUDIO_CONV] Cache hit: /tmp/tmpz0lle0cj/short_test_audio_b64fa473a841_cbr.mp3
  Result: /tmp/tmpz0lle0cj/short_test_audio_b64fa473a841_cbr.mp3
  Cache reuse: OK


In [ ]:
# Test with missing file
result = ensure_cbr_audio("/nonexistent/audio.mp3")
print(f"Missing file result: {result}")
assert result is None, "Should return None for missing file"

[AUDIO_CONV] Source file not found: /nonexistent/audio.mp3
Missing file result: None


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()